In [ ]:
initialize() {
	print("-------Sim Start-------");
	print("Initializing simulation...");
	if (!exists("asexual")){
		//parameter to toggle asexual reproduction
		defineConstant("asexual", T);
	}

	if (!exists("preload_location")){
		//parameter to toggle where to preload mutations
		defineConstant("preload_location", "mito");
	}
	if (!exists("mut_profile")){
		//parameter to toggle which mutational profile to use
		defineConstant("mut_profile", 3);
	}
	if (!exists("epi")){
		//parameter to toggle whether to include epistasis between mito and nuc muts
		defineConstant("epi", T);
	}
	if (!exists("data_file")){
		//parameter to specify file to output data to
		defineConstant("data_file", "default_fitness_over_time.txt");
	}
	if (!exists("logging")){
		defineConstant("logging", F);
	}

	defineConstant("popSize", 250);
	defineConstant("mito_chrom_length", 1e4);
	defineConstant("nuc_chrom_length", 1e6);
    
	if (!exists("num_tags")){
		defineConstant("num_tags", 20);
	}

	if (!exists("epi_rate")){
		defineConstant("epi_rate", 100);
	}
	
	//Disable pedigree tracking
	initializeSLiMOptions(keepPedigrees=F);
	
	if (!asexual){ 
		initializeSex();
	}

We initialize command line arguments to the SLiM script and provide defaults in case none are provided. These arguments are provided so automated calls to the simulation can change behavior configuration.

Default argument for "asexual" is T.
When T the simulation runs asexually. When F, the simulation runs sexually.

Default argument for "preload-location" is mito.
Load in the command line variable for the preload location. This decides which chromosome (mitochondria, or nuclear) we will seed with deleterious mutations.

Default argument for "mut_profile" is 3.
These mutations will constantly run during the simulation. This variable is called "mut_profile". If the profile is 1, we use both beneficial and deleterious mutations. If 2, we only use deleterious mutations. If 3, we only use neutral mutations.

Default argument for "epi" is T.
This will determine whether or not we use epistasis in the simulation.

Default argument for "data_file" is "default_fitness_over_time.txt".
This will determine where sim results are written to. Should nearly always be provided with a meaningful arg.

Default argument for "logging" is F.
Logging of population fitness over time in order to create representations of our data.

We establish variables for population and chromosome sizes.

A default number for the number of tags is initiates.

We define an epistasis parameter (default 100)

Disable pedigree tracking.

If "asexual"= F, we enable sex.

In [ ]:
//Synonymous muts: will experience not net fitness effects
	//or epistaic interactions. 
	initializeMutationType("m8", 0.5, "f", 0.0);
	initializeMutationType("m9", 0.5, "f", 0.0);

//create mutation distributions
	if (mut_profile == 1){
		//Both del and ben muts
		
		// Nuclear mutations
		initializeMutationType("m1", 0.5, "g", -0.01, 0.2); // Nuclear negative
		initializeMutationType("m2", 0.5, "g", 0.001, 1); // Nuclear positive

		// Mitochondrial mutations
		initializeMutationType("m3", 1.0, "g", -0.05, 0.2); // Mitochondrial negative
		initializeMutationType("m4", 1.0, "g", 0.001, 1); // Mitochondrial positive
		
		initializeGenomicElementType("g1", c(m1, m2), c(0.49, 0.01)); // Nuclear genome
		initializeGenomicElementType("g2", c(m3, m4), c(0.49, 0.01)); // Mitochondrial genome
	} else if (mut_profile == 2){
		//only del muts
		
		// Nuclear mutations
		initializeMutationType("m1", 0.5, "g", -0.01, 0.2); // Nuclear negative
		initializeMutationType("m2", 0.5, "g", -0.01, 0.2); 
				
		// Mitochondrial mutations
		initializeMutationType("m3", 1.0, "g", -0.05, 0.2); // Mitochondrial negative
		initializeMutationType("m4", 1.0, "g", -0.05, 0.2); // Mitochondrial 
	
		initializeGenomicElementType("g1", c(m1, m2), c(0.49, 0.01)); // Nuclear genome
		initializeGenomicElementType("g2", c(m3, m4), c(0.49, 0.01)); // Mitochondrial genome
	} else if (mut_profile == 3){
		//no muts (technicially synonymous muts with no fitness effect)
		//These are defined ONLY so thet they can exist in the mutation schema and not break leter code
		//Nuclear
		initializeMutationType("m1", 0.5, "f", 0.0);
		initializeMutationType("m2", 0.5, "f", 0.0);
		//Mitochondrial
		initializeMutationType("m3", 0.5, "f", 0.0);
		initializeMutationType("m4", 0.5, "f", 0.0);	

		initializeGenomicElementType("g1", c(m8), c(1.0)); // Nuclear genome
		initializeGenomicElementType("g2", c(m9), c(1.0)); // Mitochondrial genome
	}

We create two sets of mutations, one for each chromosome.

Mutations 8 and 9 are created as a control for mutations.

If the mutation profile is equal to 1, we will use both deleterious and beneficial mutations.

We have 2 mutations for the nuclear genome and 2 for the mitochondrial genome. Having specific mutations for each genome allows epistasis interactions between the genomes. All of these mutations have a distribution of gamma, to help vary their effects on the population. They also have small means to keep a minimal effect on the individuals' fitness.

Nuclear mutations: m1 is a deleterious mutation for the nuclear genome. It has a dominance coefficient of 0.5, a mean of -0.01, and an alpha shape parameter of 0.2.
m2 is a beneficial mutation. It has a dominance coefficient of 0.5, a mean of 0.001, and an alpha shape parameter of 1.

The mitochondrial mutations, m3 and m4, are nearly identical to the nuclear. m3 has a mean of -0.5, which produces a slightly greater effect on fitness.

The genomic element types create groups of mutations and proportions of those mutations. We created g1 for the nuclear mutations, and g2 for the mitochondria. These mutations can only be acquired and mutate within the bounds of the genomic element.

If the mut_profile is 2, we will only use deleterious mutations.

Both of the nuclear mutations, m1 and m2, have a dominance coefficient of 0.5, using a gamma distribution, with a mean of -0.01 and an alpha shape parameter of 0.2.

The mitochondrial mutations, m3 and m4, are slightly stronger, increasing their dominance coefficient to 1.0 and their mean to -0.5

Like the genomic elements before, these place the nuclear and mitochindria mutations in their respective groups and marking the ratios.

Profile 3 has no positive or beneficial mutations.

m1-4 are placeholders to avoid errors down the line. The mean for each is 0.0, giving them no effect. The control mutations are the ones that are actually used in the simulaton.

In [ ]:
if (preload_location == "nucl" & asexual == F){
		print("Nuclear preloading with sexual reproduction");
		initializeMutationType("m5", 0.7, "f", -0.055);
		initializeMutationType("m6", 0.7, "f", -0.052);
		initializeMutationType("m7", 0.7, "f", -0.051);
	} else if (preload_location == "nucl" & asexual == T) {
		print("Nuclear preloading with asexual reproduction");
		initializeMutationType("m5", 0.7, "f", -0.075);
		initializeMutationType("m6", 0.7, "f", -0.075);
		initializeMutationType("m7", 0.7, "f", -0.07355);
	} else {
		print("Mitochondrial preloading");
		initializeMutationType("m5", 0.7, "f", -0.075);
		initializeMutationType("m6", 0.7, "f", -0.075);
		initializeMutationType("m7", 0.7, "f", -0.07355);
	}


	m1.convertToSubstitution = F;
	m2.convertToSubstitution = F;
	m3.convertToSubstitution = F;
	m4.convertToSubstitution = F;

	m5.convertToSubstitution = F;
	m6.convertToSubstitution = F;
	m7.convertToSubstitution = F;

	m8.convertToSubstitution = F;
	m9.convertToSubstitution = F;

m5, m6, and m7 are used as preload mutations. These drop the selected chromosome into mutational meltdown, thus dropping the populations fitness to 0.0004.

These mutations are rather strong and identical for the mitochondria and asexual populations.

The nuclear sexual population drops in the second generation as it processes these mutations. As such, we decreases their mutations' strength so they stil start at 0.0004 after the drop.

Finally, we stop all mutations from converting to substitution. This keep their effect on fitness, eve after the entire population has that mutation.

In [ ]:
//chromosome 1: mitochondrial chromosome
	//adding in chromosome length
	initializeChromosome(1,mito_chrom_length, type="HF");
	initializeGenomicElement(g1, 0, mito_chrom_length-1);
	initializeMutationRate(1e-6);
	initializeRecombinationRate(0.0);
	
	//chromosome 2: autosome
	//adding chromosome length
	initializeChromosome(2,nuc_chrom_length,type="A");
	initializeGenomicElement(g2, 0, nuc_chrom_length-1);
	initializeMutationRate(1e-7);
	initializeRecombinationRate(5e-6);
}

We establish the mitochondrial chromosome, as chromosome 1, with the variable's length established  above, and the type HF. HF is a halpoid that is maternally inherited, much like the mitochondrial genome.
The genomic element g1 is added, and takes up the whole of the chromosome.
Chromosome 1 has a mutation rate of 1e-6, which is higher than nuclear's.
Chomosome 1's recombination rate is set to 0.0.

Chromosome 2 is created in the same way. It is established as an autosome instead of an HF, and uses the nuc_chrom_length established above.
G2 is initialized.
The mutation rate is lower than chromosome 1.
The recombination rate is set to 5e-6 to allow some recombination.

In [ ]:
1 early() {
	
	//initialize population
	sim.addSubpop("p1", popSize);
	//Asexual populations reproducte by cloning 100% of the time
	if (asexual == T){
		p1.setCloningRate(1.0);
	} else {
		p1.setCloningRate(0.0);
	}
	
	//Vector to store fitness over time
	p1.setValue("fitness_over_time", c());
}

"1 early()" means this is run at the beginning of the first cycle.

We initialize the population and call it "p1".

If the asexual variable is true, then we set the cloning rate to 1.0.
Otherwise, it is set to 0.0, since no one in the population is reproducting asexually.

A vector to store p1's fitness over time is estaished.

In [ ]:
//override SLiM's current fitness calculation
mutationEffect(m1) {
	return 1.0;
}

mutation(m1) {
	mut.tag = rdunif(1, 0, num_tags - 1);
	return T;
}

mutationEffect(m2) {
	return 1.0;
}

mutation(m2) {
	mut.tag = rdunif(1, 0, num_tags - 1);
	return T;
}

mutationEffect(m3) {
	return 1.0;
}

mutation(m3) {
	mut.tag = rdunif(1, 0, num_tags - 1);
	return T;
}

mutationEffect(m4) {
	return 1.0;
}

mutation(m4) {
	mut.tag = rdunif(1, 0, num_tags - 1);
	return T;
}

mutationEffect(m5) {
	return 1.0;
}

mutation(m5) {
	mut.tag = rdunif(1, 0, num_tags - 1);
	return T;
}

mutationEffect(m6) {
	return 1.0;
}

mutation(m6) {
	mut.tag = rdunif(1, 0, num_tags - 1);
	return T;
}

mutationEffect(m7) {
	return 1.0;
}

mutation(m7) {
	mut.tag = rdunif(1, 0, num_tags - 1);
	return T;
}

mutationEffect(m8) {
	return 0.0;
}

mutationEffect(m9) {
	return 0.0;
}

For each mutation, we set its mutationEffect to return 1.0. This deactivates SLiM's default mutation effect and uses our previous calculations instead. Additionaly, we add tags to each mutation.

This allows us to correctly account for epistasis.

In [ ]:
//seeds deleterious mutations
1 modifyChild() {
	//gather mitochondrial chromosomes
	
	mut_chroms = c();
	for (hap in child.haplosomes){
		if (preload_location == "mito" ){
			if (hap.chromosome.id == 1){
				mut_chroms = c(mut_chroms, hap);
			}
		} else {
			if (hap.chromosome.id == 2){
				mut_chroms = c(mut_chroms, hap);
			}
		}
	}

"1 modifyChild()" modifies the offspring for the first generation.

To begin we create a vector to hold the chromosomes we wish to modify.
For each haplsome in each child's genome (this includes the two haplosomes that make up chromosome 2), if the variable preload_location is "mito" we gather the haplosomes of chromosome 1 into the list.
If it is "nucl", we gather the same from chromosome 2.

In [ ]:
if (preload_location == "mito") {
		seed_len = mito_chrom_length;
	} else {
		seed_len = nuc_chrom_length;
	}

	for (i in 1:20){
		newMuts = mut_chroms.addNewDrawnMutation(m5, rdunif(1, 0, seed_len - 1));
		for (newMut in newMuts)
			newMut.tag = rdunif(1, 0, num_tags - 1);
		newMuts = mut_chroms.addNewDrawnMutation(m6, rdunif(1, 0, seed_len - 1));
		for (newMut in newMuts)
			newMut.tag = rdunif(1, 0, num_tags - 1);
		newMuts = mut_chroms.addNewDrawnMutation(m7, rdunif(1, 0, seed_len - 1));
		for (newMut in newMuts)
			newMut.tag = rdunif(1, 0, num_tags - 1);
	}
	
	
	return T;
}

Depending on the pre-load location, we seed each of the pre-established deleterious mutations, m5-7, into the chosen genome 20 times, leading to 60 total mutations.

Finally, we return True to allow all of the low fitness offspring into the simulation.

In [ ]:
1: late() {
	
	
	inds = sim.subpopulations.individuals;
	popFitt = c();
	
	// loop through each individual in the population to calculate fitness
	for (ind in inds) {
		globalFitness = 0;
		
		unique = ind.uniqueMutations; // get the unique mutations for the individual
		
		if (epi == F) {
			for (m in unique) {
				globalFitness = globalFitness + (m.selectionCoeff * abs(m.selectionCoeff));
			}
		} else {
			if (preload_location == "mito") {
				uniqueNuc = unique[unique.mutationType == m1 | unique.mutationType == m2]; 
				uniqueMito = unique[unique.mutationType == m3 | unique.mutationType == m4 | unique.mutationType == m5 | unique.mutationType == m6 | unique.mutationType == m7];
			} else {	
				uniqueNuc = unique[unique.mutationType == m1 | unique.mutationType == m2 | unique.mutationType == m5 | unique.mutationType == m6 | unique.mutationType == m7]; 
				uniqueMito = unique[unique.mutationType == m3 | unique.mutationType == m4];
			}

At the end of each generation, we calculate the fitness for the populations.

First, we gather all of the individuals and locate the unique mutations for each individual.

In [ ]:
// loop through each mitochondrial mutation to calculate the epistatic interactions with nuclear mutations
			for (m in uniqueMito) {
				
				
				total = 0;
				epistatic = uniqueNuc[uniqueNuc.position % epi_rate == m.position % epi_rate]; // get the nuclear mutations that are epistatic with the mitochondrial mutation (i.e., those that occur at the same position modulo 125)
				
				//UPGRADE
				// loop through the epistatic nuclear mutations to calculate the total effect on fitness
				for (n in epistatic) {
					if (n.tag == m.tag) {
						total = total + abs(n.selectionCoeff);
					}
					else {total = total - abs(n.selectionCoeff);}
				}
				
				if (length(epistatic) == 0) {
					globalFitness = globalFitness + (m.selectionCoeff * abs(m.selectionCoeff));
				}
				else {
					globalFitness = globalFitness + (total * abs(m.selectionCoeff));
				}

We loop through each unique mutation in the mitochondrial genome.
If epi has not been activated, we add to the global fitness the selection coefficient of the mutation squared while preserving its sign. 
If epi has been activated, then we gather all potentially epistatic nuclear mutations as determined by the epiSt argument. Mutations are epistatic if the position of the nuclear mutation % epi_rate is equal to the position of the mitochondrial mutation % epi_rate.

In [ ]:
// loop through the nuclear mutations to add their effects on fitness (after accounting for epistasis with mitochondrial mutations)
			for (mut in uniqueNuc) {
				globalFitness = globalFitness + (abs(mut.selectionCoeff) * mut.selectionCoeff);
			}
		}

Likewise, we loop through the epistatic nuclear mutations to calculate the total effect on fitness. 

The two mutations' fitness levels are mutliplied together and if their tags match, the product is added to the individual's fitness. If the tags don't match, the product is subtracted.

In [ ]:
// calculate the final fitness for the individual, ensuring it is not negative
		Fitness = max(0.0, 1.0 + globalFitness * 3.0); // scale fitness to ensure it is positive and has a reasonable range
		popFitt = c(popFitt, Fitness);
	}
	
	inds.fitnessScaling = popFitt;
}

For each individual, we scale their individual fitness according to a linear fitness to ensure that the majority of fitness values are between 0 and 2. 

We then add the individual fitness to an array of all individuals' fitness. FitnessScaling is applied to each individual.

In [ ]:
2 early() {
	//Mutational metdown fitness
	print("Mean fitness at generation 1:");
	print(mean(p1.cachedFitness(NULL)));
    
//tracks how far down nucl sexual sinks after being seeded
3 early() {
	//Mutational metdown fitness
	print("Mean fitness at generation 2:");
	print(mean(p1.cachedFitness(NULL)));
}

At the beginning of generation 2, we print out the mean fitness of the population.

At the beginning of generation 3, we print out the mean fitness as well. This block is specifically for nuclear sexual, since it's fitness hits it's lowest here.

In [ ]:
2 early() {
	//Mutational metdown fitness
	print("Mean fitness at generation 1:");
	print(mean(p1.cachedFitness(NULL)));
}

//tracks how far down nucl sexual sinks after being seeded
3 early() {
	//Mutational metdown fitness
	print("Mean fitness at generation 2:");
	print(mean(p1.cachedFitness(NULL)));
}

This segment tracks the starting point of the sim, after the preloading.

In [ ]:
2001 early() {
	//Final fitness - output
	print("Mean fitness at generation 2000:");
	print(mean(p1.cachedFitness(NULL)));
}

At the beginning of the 2001 generation, we print the final fitness.

In [ ]:
2:2001 early() {
	//update stored data about fitness over time for this simulation
	f = mean(p1.cachedFitness(NULL));
	
	v = p1.getValue("fitness_over_time");
	v = c(v, f);
	p1.setValue("fitness_over_time", v);
}

For the beginning of each generation from 2 - 2001, we update the mean fitness, and add it to the "fitness_over_time" vector.

In [ ]:
2001 late() {
	//Export fitness over time data for this simulation
	v = p1.getValue("fitness_over_time");
	s = paste(v, sep=" ");
	if (asexual == T)
		writeFile(getwd() + "/asexual_fitness_over_time.txt", s, append=T);
	else
		writeFile(getwd() + "/sexual_fitness_over_time.txt", s, append=T);
	
	print(length(v) + " entries exported");
	print("Simulation complete");
}

At the end of the 2001 generation, we write the "fitness_over_time" vector to its respective file, either asexual or sexual.

We then print the number of entries exported and that the simulation is complete.